Langchain, Huggignface Models are Used.<br/>

Chatbot - Ex: Travel Assistant. <br/>
Model used - TinyLlama/TinyLlama-1.1B-Chat-v1.0 <br/>
Chat History is chained to give a Convrsational feeling and History Content for the Assistant.<br/>

Output of Chatbot is Translated to French. <br/>
Model used - Helsinki-NLP/opus-mt-en-fr <br/>
Translation is only to print the output.

In [ ]:
# Import Sys Packages
import os
import keyboard
from dotenv import load_dotenv

#Import SpaCy - No Used 
#import spacy
#nlp_spacy = spacy.load("en_core_web_sm")

# Import Langchain Huggingface Chatbot Interface
from langchain_huggingface import ChatHuggingFace
from langchain_huggingface import HuggingFacePipeline
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# Load 'env' and set OpenAI API Key
load_dotenv()


True

In [44]:
# Instantiate ChatHuggignFace LLM for Chatbot 
hf_chat_llm = HuggingFacePipeline.from_model_id(
              model_id="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
              task="text-generation",
              device=-1,
              pipeline_kwargs={
                  "max_new_tokens":1000,
                  "temperature":0.9,
                  "top_p":0.8
              }
            )

chatbot = ChatHuggingFace(llm=hf_chat_llm)

Device set to use cpu


In [45]:
# Instantiate Tranlslation LLM for Chatbot.
hf_trans_en_fr_llm = HuggingFacePipeline.from_model_id(
                   model_id = "Helsinki-NLP/opus-mt-en-fr",
                   task="translation",
                   device=-1,
                   pipeline_kwargs={
                     "max_new_tokens": 1000
                  }
                )

c:\Products\Anaconda3\envs\hf1_env\Lib\site-packages\transformers\models\marian\tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
Device set to use cpu


In [46]:
# Format Chat Chain

def format_system_messages(chat_hist, sys_msg):
    chat_hist = [
        SystemMessage(content=sys_msg)
    ]
    return(chat_hist)

def format_user_messages(chat_hist, usr_msg):
    chat_hist.append(HumanMessage(content=usr_msg))
    return(chat_hist)

def format_assistant_messages(chat_hist, ai_msg):
    chat_hist.append(AIMessage(content=ai_msg))
    return(chat_hist)

In [47]:
#Chat with AIAssistant

def chat(chat_hist):
    response = chatbot.invoke(chat_hist)
    print(response.content)
    ai_message = response.content.split("<|assistant|>")[-1].strip()
    return(ai_message)

In [ ]:
# Translate Chat to French
# Line by line of respnse sentece from LLM. 

def translate(trans_input):
    sentence_in = []
    sentence_out = []
    sentence_in = trans_input.splitlines()     # Split Sentence on '\n'
    
    for text in sentence_in:
        if (text != ''):
            text_out = hf_trans_en_fr_llm.invoke(text)
            sentence_out.append(text_out)

    trans_out = ''
    for text in sentence_out:                  # Concatinate Array with '\n'
        trans_out = trans_out + text + '\n\n'

    return(trans_out)


In [ ]:
# Initiate Chat with AI

chat_chain = []
exit_flag = False

def on_key_press(event):
    if event.name == 'esc':
        print("Escape key pressed! Exiting input.")
        global exit_flag
        exit_flag = True
        return True  # Stop the keyboard listener

# Keyboard Listener
keyboard.on_press(on_key_press)

# System Prompt - Set teh Context for the bot.
system_prompt = input("What type Assistant should i be today: ")
chat_chain = format_system_messages(chat_chain, system_prompt)

while(not exit_flag):
    user_prompt = input("User: ")
    if (not exit_flag):
        chat_chain = format_user_messages(chat_chain, user_prompt)
        ai_response = chat(chat_chain)          # Sent to LLM Chat Interface.
        fr_response = translate(ai_response)    # Sent to LLM Translate Interface.
        print("Assistant French:\n", fr_response)
        chat_chain = format_assistant_messages(chat_chain, ai_response)


<|system|>
You are a helpful Travel Assistant</s>
<|user|>
Information about top 10 things to see in Spain</s>
<|assistant|>
1. Madrid: Madrid is the capital city of Spain and home to the famous Plaza de España, the Plaza Mayor, and the Royal Palace of Madrid. It's also known for its traditional tapas bars, such as El Celler de Can Roca and El Celler de Can Gava.

2. Barcelona: Barcelona is the second-largest city in Spain and home to the famous Sagrada Familia, Park Guell, and the La Rambla street. It's also known for its vibrant nightlife, including the famous Barceloneta beach and the beach clubs like Groove.

3. Valencia: Valencia is a city on the Mediterranean coast of Spain and home to the beautiful and historic University of Valencia. It's also known for its traditional tapas bars, such as Txikito and El Ampurdán.

4. Seville: Seville is the capital city of Andalusia and home to the famous Almadén de los Monteros, the Alcázar of Seville, and the Cathedral of Seville. It's also k